# Train Sign Language Recognition Model

This notebook trains a neural network to recognize hand gestures from your collected dataset.

## What you'll learn:
- Loading data from CSV
- Preprocessing features and labels
- Training an MLP neural network
- Evaluating model performance
- Saving the trained model

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import os

print("Libraries imported successfully!")

: 

## 2. Load the Dataset

The dataset contains 63 features (21 hand landmarks × 3 coordinates) plus a label column.

In [ ]:
# Check if dataset exists
dataset_path = "dataset/data.csv"
if not os.path.exists(dataset_path):
    print(f"Error: Dataset not found at {dataset_path}")
    print("Please run collect_dataset.py first to collect data.")
else:
    # Load the dataset
    data = pd.read_csv(dataset_path, header=None)
    print(f"Dataset loaded: {len(data)} samples")
    print(f"Features: {data.shape[1]} columns (63 landmarks + 1 label)")
    
    # Show first few rows
    print("\nFirst 5 rows:")
    data.head()

## 3. Prepare Features and Labels

- **X**: All columns except the last (the 63 landmark coordinates)
- **y**: The last column (gesture label like 'hello', 'yes', etc.)

In [ ]:
# Split into features (X) and labels (y)
X = data.iloc[:, :-1].values  # All columns except last
y = data.iloc[:, -1].values   # Last column only

print(f"Feature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"\nUnique classes: {list(set(y))}")
print(f"Number of classes: {len(set(y))}")

## 4. Encode Labels

Convert text labels ('hello', 'yes', etc.) to numbers (0, 1, 2, ...)

In [ ]:
# Create label encoder
le = LabelEncoder()

# Fit and transform labels
y_encoded = le.fit_transform(y)

# Show the mapping
print("Label mapping:")
for i, label in enumerate(le.classes_):
    print(f"  {label} → {i}")

## 5. Split Train and Test Sets

80% for training, 20% for testing

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.2,    # 20% for testing
    random_state=42   # For reproducible results
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## 6. Create and Train the Model

We use an MLP (Multi-Layer Perceptron) neural network with:
- 2 hidden layers (128 and 64 neurons)
- ReLU activation
- Up to 500 training iterations

In [ ]:
# Create the model
model = MLPClassifier(
    hidden_layer_sizes=(128, 64),  # Two hidden layers
    max_iter=500,                  # Maximum iterations
    random_state=42,               # Reproducible results
    verbose=True                   # Show training progress
)

print("Training the model...")
model.fit(X_train, y_train)
print("\nTraining complete!")

## 7. Evaluate the Model

In [ ]:
# Make predictions on test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

# Detailed report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 8. Confusion Matrix

See which gestures get confused with each other

In [ ]:
# Display confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Show with class labels
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 9. Save the Model

Save the trained model and label encoder for use in the real-time app

In [ ]:
# Save the model
joblib.dump(model, "model.pkl")
print("Model saved: model.pkl")

# Save the label encoder
joblib.dump(le, "labels.pkl")
print("Label encoder saved: labels.pkl")

print("\n✅ Training complete! You can now run: python updated_realtime_test.py")